# Transpose, Submatrices, and Properties

Every matrix has a mirror image. Flip it across its main diagonal and the rows
become columns; that single move is the **transpose**, and it is the last piece of
notation this section owes you. It is also the one you will meet most often
afterwards: the dagger that reverses a quantum gate is this flip plus one extra
step, the rule for undoing a sequence of gates is this flip's order-reversing
property, and the block structure of a controlled gate is a submatrix sitting in
the corner of a bigger matrix.

This notebook does three things. It defines the flip and drills it until the shape
change is automatic. It states the five properties of the flip: the three that
push a transpose through scalars, sums and products, the one that undoes it, and
the one that defines a symmetric matrix. And it ends where matrix algebra stops
behaving like ordinary arithmetic: two laws you are entitled to expect from
numbers that matrices simply do not obey.

**Objectives:**
- Transpose a matrix by hand, including the rectangular case where the shape changes
- State and verify the five transpose properties, and explain why the product rule reverses the order
- Recognize a symmetric matrix, and build one from any matrix in two different ways
- Extract a submatrix two ways: by deleting a row and a column, and by slicing a block
- Name two laws of ordinary arithmetic that matrices break, with a counterexample for each

**Reference:** See [`../GUIDE.md`](../GUIDE.md).

<!-- browser-runnable -->

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

## 1. The transpose is a flip across the main diagonal

The **main diagonal** of a matrix runs from the top-left entry down and to the
right: row 1 column 1, then row 2 column 2, then row 3 column 3. The **transpose**
is what you get by flipping the whole matrix across that line. Entries sitting on
the diagonal do not move at all. Every other entry trades places with its mirror
partner on the far side.

Three descriptions of the same move, and it is worth carrying all three:

- **The fold.** Hold the main diagonal fixed and fold the matrix over it, the way
  a book closes along its spine.
- **Rows become columns.** Row 1 of the original becomes column 1 of the flipped
  matrix, row 2 becomes column 2, and so on. This is the description to use when
  you are computing one by hand.
- **The index swap.** Whatever sits in row i, column j ends up in row j, column i.

Take the small square case first, on paper, before you run anything:

$$
\begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}
\longrightarrow
\begin{pmatrix} 1 & 3 \\ 2 & 4 \end{pmatrix}
$$

The 1 and the 4 sit on the diagonal and stay where they are. The 2 was in row 1,
column 2 and lands in row 2, column 1; the 3 makes the opposite trip. In a two by
two, nothing else can happen.

A square matrix keeps its shape, which makes it easy to miss what actually
happened. A rectangular one leaves no doubt: a matrix with 2 rows and 3 columns
flips into a matrix with 3 rows and 2 columns. The row count and the column count
trade places, always — and a shape mismatch further down your work is very often a
flip you forgot to take.

In [ ]:
M2 = np.array([[1, 2],
               [3, 4]])
print('M2 =')
print(M2)
print('M2 flipped =')
print(M2.T)                       # 1 and 4 held; 2 and 3 swapped

M23 = np.array([[1, 2, 3],
                [4, 5, 6]])
print('\nM23 shape:', M23.shape, '-> flipped shape:', M23.T.shape)
print(M23.T)

colv = np.array([[7],
                 [8],
                 [9]])            # a genuine column: 3 rows, 1 column
print('\ncolv shape:', colv.shape, '-> flipped shape:', colv.T.shape)
print(colv.T)

flat = np.array([7, 8, 9])        # 1-D: there is no second axis to flip
print('\nflat shape:', flat.shape, '-> flipped shape:', flat.T.shape)

Read the last two blocks of output together, because they are the trap.

`np.array([7, 8, 9])` is **not** a column vector. It is a one-dimensional array —
it has a length and nothing else — so flipping it returns exactly what went in,
with no error and no warning. NumPy is not being unhelpful; there is genuinely no
second axis to fold over.

A column vector is a two-dimensional array with one column, written
`[[7], [8], [9]]` or built with `np.array([7, 8, 9]).reshape(3, 1)`. Only then
does the flip produce the row `[[7, 8, 9]]`, shape `(1, 3)`, still two-dimensional.
Whenever a transpose appears to do nothing, print `.shape` first: a shape with a
single number in it, like `(3,)`, is the whole explanation.

**Notation.** The transpose of $M$ is written $M^{T}$ — a capital T set as a
superscript, never a multiplication. The index rule is

$$
(M^{T})_{ij} = M_{ji}
$$

read as "the entry of the flipped matrix in row i, column j is the entry of the
original in row j, column i". In NumPy it is `M.T`, and `M.transpose()` is the
same thing spelled out.

## 2. Five properties of the transpose

Five facts carry all the transpose algebra you will ever need. Each is stated in
plain English first, because the symbolic version is unreadable until you already
know what it claims. One small pair does duty as the hand case for all of them:

$$
P = \begin{pmatrix} 2 & 1 \\ 4 & 3 \end{pmatrix},
\qquad
Q = \begin{pmatrix} 1 & 5 \\ 0 & 2 \end{pmatrix}
$$

Flip each one now, because both flips are needed repeatedly below:

$$
P^{T} = \begin{pmatrix} 2 & 4 \\ 1 & 3 \end{pmatrix},
\qquad
Q^{T} = \begin{pmatrix} 1 & 0 \\ 5 & 2 \end{pmatrix}
$$

### Property 1 — flipping twice changes nothing

Fold the matrix over the diagonal, then fold it back: every entry returns to where
it started. Flipping P gives `[[2, 4], [1, 3]]`; flipping that gives
`[[2, 1], [4, 3]]`, which is P again. In symbols,

$$
(A^{T})^{T} = A
$$

### Property 2 — a scalar factor passes straight through

Multiplying every entry by a number does not care where in the grid the entries
sit, so scaling and flipping never interfere. Take c = 3. Scaling first gives
`[[6, 3], [12, 9]]`, whose flip is `[[6, 12], [3, 9]]`. Flipping first gives
`[[2, 4], [1, 3]]`, and scaling that by 3 gives `[[6, 12], [3, 9]]` — the same
matrix, reached the other way round. In symbols,

$$
(cA)^{T} = c A^{T}
$$

In [ ]:
P = np.array([[2, 1],
              [4, 3]])
Q = np.array([[1, 5],
              [0, 2]])

print('P flipped twice equals P?', np.allclose(P.T.T, P))

print('\n(3P) flipped =')
print((3 * P).T)
print('3 * (P flipped) =')
print(3 * P.T)
print('same matrix?', np.allclose((3 * P).T, 3 * P.T))

### Property 3 — the transpose of a product reverses the order

This is the property that surprises people, and the one worth memorizing on its
own: **flip a product and the factors come back out backwards.**

$$
(AB)^{T} = B^{T} A^{T}
$$

Work it by hand on the pair above. The product is `P @ Q = [[2, 12], [4, 26]]`, so
its flip is `[[2, 4], [12, 26]]`. Now the reversed product of the flipped factors:
`Q.T @ P.T` multiplies `[[1, 0], [5, 2]]` by `[[2, 4], [1, 3]]`, and row 1 of the
first against the two columns of the second gives 2 and 4, row 2 gives 12 and 26.
That is `[[2, 4], [12, 26]]` — the same matrix.

Now the version you would write if you were guessing, keeping the order as it was:
`P.T @ Q.T` comes out `[[22, 8], [16, 6]]`. Not close. Not a rounding difference,
not a sign slip — a completely different matrix. Keeping the order is not a small
error, and this counterexample is worth remembering as the reason the rule is
stated so insistently.

**Why the reversal is forced, not a convention.** Suppose A has 2 rows and 3
columns and B has 3 rows and 4 columns. Then AB has 2 rows and 4 columns, so
$(AB)^{T}$ has 4 rows and 2 columns. Meanwhile $A^{T}$ is 3 by 2 and $B^{T}$ is 4
by 3. The product $A^{T}B^{T}$ would need the inner dimensions to agree, 2 against
4, and they do not — that product does not exist at all. But $B^{T}A^{T}$ is 4 by
3 times 3 by 2, which is 4 by 2: exactly the shape required. Only one order can
work. The rule does not choose it; the shapes do.

The everyday version is socks and shoes. To undo "socks, then shoes" you take off
the shoes first and the socks second. Reversing a sequence reverses its steps.

In [ ]:
print('(P @ Q) flipped =')
print((P @ Q).T)
print('Q.T @ P.T =')
print(Q.T @ P.T)
print('rule holds?', np.allclose((P @ Q).T, Q.T @ P.T))

# Keep the order instead of reversing it and the answer is something else entirely.
print('\nP.T @ Q.T =')
print(P.T @ Q.T)
print('same as (P @ Q) flipped?', np.allclose((P @ Q).T, P.T @ Q.T))

### Property 4 — sums pass straight through

Addition is entrywise and flipping only moves entries around, so the two never
interfere. Adding then flipping and flipping then adding give the same matrix:

$$
(A + B)^{T} = A^{T} + B^{T}
$$

Hand check: `P + Q = [[3, 6], [4, 5]]`, whose flip is `[[3, 4], [6, 5]]`. Going the
other way, `P.T + Q.T` adds `[[2, 4], [1, 3]]` to `[[1, 0], [5, 2]]` and gives
`[[3, 4], [6, 5]]`. Agreed.

Notice what property 4 does **not** say. There is no reversal here. Addition does
not care about order in the first place, so there is no order to undo; matrix
multiplication does care, which is exactly why property 3 has to reverse and
property 4 does not.

Subtraction comes free, and it is a **corollary** rather than a sixth
property: $A - B$ is $A + (-1)B$, so properties 4 and 2 together give
$(A - B)^{T} = A^{T} - B^{T}$ with nothing new to remember.

### Property 5 — a matrix that is its own flip is symmetric

$$
A = A^{T}
$$

That equation is not a theorem to prove; it is the **definition** of a symmetric
matrix. In words: the fold across the main diagonal leaves the matrix unchanged,
so every entry equals its mirror partner — the entry in row 1, column 2 equals the
one in row 2, column 1, and so on for every pair. The diagonal itself is free to
be anything, since those entries are their own partners.

Only a square matrix can be symmetric. A 2 by 3 matrix flips into a 3 by 2, and
two matrices of different shapes cannot be equal, so the question is closed before
you look at a single entry.

In [ ]:
print('(P + Q) flipped =')
print((P + Q).T)
print('P.T + Q.T =')
print(P.T + Q.T)
print('rule holds?', np.allclose((P + Q).T, P.T + Q.T))

sym_demo = np.array([[2, 7],
                     [7, 5]])
notsym_demo = np.array([[2, 7],
                        [1, 5]])
print('\nsym_demo symmetric?   ', np.allclose(sym_demo, sym_demo.T))
print('notsym_demo symmetric?', np.allclose(notsym_demo, notsym_demo.T))

## 3. Two constructions that are always symmetric

Symmetric matrices are worth naming because they are unreasonably well behaved:
half the matrix determines the other half, so half the entries carry all the
information and any argument you make about one mirror pair covers both. A great
deal of applied mathematics consists of noticing that some matrix is symmetric
and then relaxing. Covariance matrices in statistics are symmetric; so are the
energy matrices of physics and chemistry, whose complex-valued cousins arrive in
`00-prereqs` under the name Hermitian.

Two recipes turn any matrix into a symmetric one, and the properties above prove
both in a single line each.

**The sum with its own flip.** For any square A, the matrix $A + A^{T}$ is
symmetric. Flip it and use property 4, then property 1:

$$
(A + A^{T})^{T} = A^{T} + (A^{T})^{T} = A^{T} + A = A + A^{T}
$$

The last step is just the fact that addition does not care about order. The result
is its own flip, which is the definition.

**The product with its own flip.** For any A at all — square or not — the matrix
$A A^{T}$ is symmetric. Flip it and use property 3, then property 1:

$$
(A A^{T})^{T} = (A^{T})^{T} A^{T} = A A^{T}
$$

Note where the reversal went: property 3 sends the flipped second factor to the
front, and that flipped-flipped factor is just A again. The reversal is what makes
this work; without it the argument would not close.

**One normalized variant, mentioned once.** Halving the first recipe gives
$(M + M^{T})/2$, which is symmetric for exactly the same reason and averages
each mirror pair instead of doubling it — handy when the entries carry units you
want to preserve. This notebook stays with $A + A^{T}$ and $A A^{T}$ because both
keep every entry a whole number.

The rectangular case is worth watching in the output below. If A has 2 rows and 3
columns then $A^{T}$ has 3 rows and 2 columns, so $A A^{T}$ is 2 by 2 — square,
even though A was not. That is why this construction shows up whenever someone
needs a square, well-behaved matrix built out of data that arrived in a rectangle.

You have just proved two theorems using rules you learned five minutes ago. That
is what the properties are for: they let you reason about flips without computing
a single entry.

In [ ]:
print('P + P.T =')
print(P + P.T)
print('symmetric?', np.allclose(P + P.T, (P + P.T).T))

print('\nP @ P.T =')
print(P @ P.T)
print('symmetric?', np.allclose(P @ P.T, (P @ P.T).T))

# Rectangular input, square output: 2 rows by 3 columns gives a 2 by 2 result.
print('\nM23 @ M23.T =')
print(M23 @ M23.T)
print('shape:', (M23 @ M23.T).shape,
      ' symmetric?', np.allclose(M23 @ M23.T, (M23 @ M23.T).T))

## 4. Submatrices, two ways — and two conventions that disagree

A **submatrix** is what is left of a matrix when you keep some of its rows and
some of its columns and throw the rest away. You will meet submatrices constantly:
determinants are defined through them, block structure in a big matrix is read
through them, and every time you pull a corner out of an array in NumPy you are
taking one.

There are two ways to say which submatrix you mean, they come from two different
traditions, and **they count differently**. Learning them side by side now is much
cheaper than discovering the mismatch inside a bug.

**The textbook way: delete a row and a column.** "Delete row 2 and column 3" names
what goes away, and counts from **one** — the first row is row 1. Deleting one row
and one column from a 3 by 3 leaves a 2 by 2, and that little matrix is called a
**minor**; it is the building block of determinants and cofactors. In NumPy,
`np.delete(A, i, axis=0)` drops a row and `np.delete(A, j, axis=1)` drops a column,
but both take zero-based positions, so a textbook "row 2" arrives as `1`.

**The NumPy way: slice the block you want.** `A[0:2, 1:3]` names what you **keep**,
counts from **zero**, and treats the second number of each range as a stop that is
**not** included — half-open, exactly like `range`. So `1:3` keeps positions 1 and
2, which a textbook would call columns 2 and 3.

Both conventions are stacked in the cell below on the same little matrix, so you
can watch a single index mean two different things.

In [ ]:
G = np.array([[2, 9, 4],
              [7, 5, 3],
              [6, 1, 8]])
print('G =')
print(G)

# Textbook: delete row 2 and column 3, counting from one.
# In NumPy those are positions 1 and 2, counting from zero.
minor_23 = np.delete(np.delete(G, 1, axis=0), 2, axis=1)
print('\nminor after deleting row 2 and column 3 =')
print(minor_23)

# NumPy slice: keep rows 0 and 1, columns 1 and 2 -- the stop index is excluded.
print('\nG[0:2, 1:3] =')
print(G[0:2, 1:3])

# A 4 by 4 with a 2 by 2 block sitting in its bottom-right corner.
block_demo = np.array([[1, 0, 0, 0],
                       [0, 1, 0, 0],
                       [0, 0, 0, 1],
                       [0, 0, 1, 0]])
print('\nbottom-right 2 by 2 block =')
print(block_demo[2:4, 2:4])

The two conventions, laid against each other:

| Question | Textbook (minor) | NumPy (slice) |
|---|---|---|
| The first row is called | row 1 | row 0 |
| You name | what to delete | what to keep |
| A range like "2 to 3" | includes both ends | `1:3` keeps 1 and 2, stops before 3 |
| The result of naming one row and one column | one row and one column vanish | one row and one column survive |

Two habits keep this from biting. First, say out loud which convention you are in
before you write an index — "textbook row 2" and "NumPy row 2" are different rows.
Second, check the shape of what came back: deleting one row and one column from a
3 by 3 must leave a 2 by 2, and a slice `r0:r1` must return exactly `r1 - r0` rows.
A shape you did not expect is the cheapest possible bug report.

One more thing worth noticing in that last block of output. The 4 by 4 matrix
carries the 2 by 2 matrix `[[0, 1], [1, 0]]` in its bottom-right corner, and the
top-left corner is a 2 by 2 identity. That layout is not decoration — it is how a
**controlled** operation is written, and the closer at the end of this notebook
names the gate you were looking at.

## 5. Laws that hold, and two famous ones that fail

Matrix algebra keeps most of the rules of ordinary arithmetic:

- **Associativity.** $(AB)C = A(BC)$ — you may regroup a chain of products.
- **Distributivity.** $A(B + C) = AB + AC$, and $(A + B)C = AC + BC$.
- **The identity behaves.** $IA = AI = A$, for identity matrices of the right size.
- **Scalars slide.** $c(AB) = (cA)B = A(cB)$.
- **The five transpose properties** above.

It also breaks rules you would never think to doubt. One you have already met:
$AB$ and $BA$ are usually different matrices, and are often not even the same
shape. Two more, and these are the ones that catch people mid-proof:

**Cancellation fails.** With ordinary numbers, if a is not zero and
$ab = ac$, then b = c: divide both sides by a. With matrices, $AB = AC$ and A
nonzero does **not** force B = C. The counterexample below uses the matrix
`[[1, 0], [0, 0]]`, which keeps the first row of whatever it multiplies and wipes
the second. Two matrices that agree in their first row and differ in their second
become indistinguishable the moment it touches them.

**A zero product does not need a zero factor.** With numbers, $ab = 0$ means a is
zero or b is zero. With matrices, two perfectly nonzero matrices can multiply to
the zero matrix — the same eraser above, times a matrix that keeps only the second
row, wipes everything.

The single idea behind both failures: **you cannot divide by a matrix.** Some
matrices destroy information (the eraser above throws a whole row away), and there
is no operation that puts it back. When a matrix *can* be undone — when it has an
inverse — cancellation returns and both laws hold again. Inverses, determinants
and eigenvalues are not part of this module, which is four notebooks long and
ends with this one; what the four give you is the arithmetic all three of those
subjects are built on.

In [ ]:
erase = np.array([[1, 0],
                  [0, 0]])          # keeps row 1, wipes row 2
mat_b = np.array([[1, 2],
                  [3, 4]])
mat_c = np.array([[1, 2],
                  [9, 9]])          # same first row, different second row

print('erase @ mat_b =')
print(erase @ mat_b)
print('erase @ mat_c =')
print(erase @ mat_c)
print('products equal?', np.allclose(erase @ mat_b, erase @ mat_c))
print('but mat_b equals mat_c?', np.allclose(mat_b, mat_c))

keep_second = np.array([[0, 0],
                        [0, 1]])    # keeps row 2, wipes row 1
print('\nerase @ keep_second =')
print(erase @ keep_second)
print('either factor zero?',
      np.allclose(erase, 0) or np.allclose(keep_second, 0))

## 6. Notation cheat sheet

| Math | NumPy | Meaning |
|---|---|---|
| $M^{T}$ | `M.T` | Transpose: rows become columns |
| $(M^{T})_{ij} = M_{ji}$ | `M.T[i, j] == M[j, i]` | The index swap, entry by entry |
| $(A^{T})^{T} = A$ | `np.allclose(A.T.T, A)` | Flipping twice changes nothing |
| $(cA)^{T} = cA^{T}$ | `np.allclose((c * A).T, c * A.T)` | Scalars pass through |
| $(A + B)^{T} = A^{T} + B^{T}$ | `np.allclose((A + B).T, A.T + B.T)` | Sums pass through |
| $(AB)^{T} = B^{T}A^{T}$ | `np.allclose((A @ B).T, B.T @ A.T)` | Products reverse |
| $A = A^{T}$ | `np.allclose(A, A.T)` | The definition of symmetric |
| minor | `np.delete(np.delete(A, i, 0), j, 1)` | Drop one row and one column |
| block | `A[r0:r1, c0:c1]` | Keep a rectangular block |

Four habits worth taking with you:

1. **Check the shape after every flip.** The row count and column count trade
   places; if they did not, you flipped something other than what you meant to.
2. **Reverse the order in a product, every time.** $B^{T}A^{T}$, never
   $A^{T}B^{T}$. When you cannot remember why, count the shapes and let them
   remind you.
3. **Say which convention you are in** before writing an index. Textbook rows
   start at 1 and name what is deleted; NumPy rows start at 0 and name what is
   kept, with the stop excluded.
4. **Never cancel a matrix.** $AB = AC$ does not give B = C, and $AB = 0$ does not
   give a zero factor. Reach for those steps and you have assumed an inverse you
   may not have.

## 7. Drills and exercises

Flipping and slicing are muscle memory, and muscle memory needs repetition rather
than understanding. The drill helper hands you an endless supply of small problems
so the arithmetic stops costing you attention.

`d.show()` prints a fresh problem. `d.check(your_answer)` tells you which entries
are off and by how much — never what they should be. `d.reveal()` is the only
thing that prints the answer, and it is there for when you are genuinely stuck
rather than for when you are impatient.

Two kinds are fair game after this notebook: `"transpose"` and `"submatrix"`.
Levels run 1 to 3 — level 1 is small non-negative two by twos, level 2 brings
negative entries and three by threes, level 3 brings rectangles. Passing a `seed`
reproduces a problem exactly; leave it out and the module draws one and prints the
seed it used, so you can come back to any problem that beat you.

Below the drills, ten exercises. Each has two hint tiers — open only what you need
— and a check cell that tells you when you have it. Worked solutions wait at the
bottom of the notebook; run the checks before you look.

In [ ]:
from lib.linalg_drills import drill

# A transpose drill. Change the kind, the level, or the seed and run it again.
d_flip = drill("transpose", level=2, seed=8)
d_flip.show()

# A submatrix drill. Level 3 brings rectangular shapes and non-adjacent picks.
d_block = drill("submatrix", level=3, seed=4)
d_block.show()

# To answer one, write your rows out and hand them to check(). Uncomment the two
# lines below and replace the dots with your own numbers, one list per row:
#   my_rows = [[..., ...], [..., ...]]
#   d_flip.check(my_rows)
# check() reports which entries are off, and by how much, and nothing else.
#   d_flip.reveal()   # the only path to the answer

### Exercise 1 — Flip a rectangle by hand

Take

$$
E = \begin{pmatrix} 3 & 1 & 4 \\ 2 & 7 & 5 \end{pmatrix}
$$

and write its transpose down on paper before you type anything. E has 2 rows and 3
columns, so decide what shape the answer must have before you place a single
entry.

Define `ex1_t` — the transpose of E, either as a nested list of rows or as a NumPy
array. Type the entries in yourself rather than calling `.T` on E; the flip is the
exercise, and the shortcut is what you are trying to make unnecessary.

<details><summary>Hint 1 — nudge</summary>

Row 1 of E becomes column 1 of the answer, and row 2 becomes column 2. Since E has
only two rows, the answer has only two columns — and one row for each of E's three
columns.

</details>
<details><summary>Hint 2 — approach</summary>

Write the answer as three rows of two numbers each. The first row of your answer
holds the first entry of E's row 1 followed by the first entry of E's row 2 — that
is, E's first column, laid out sideways. Do the same for E's second and third
columns to get the remaining two rows.

</details>

In [ ]:
# Exercise 1: Flip the 2-by-3 matrix E across its main diagonal by hand.
# Define: ex1_t

# TODO: your code here

In [ ]:
# Check Exercise 1 -- run after your attempt.
from lib.grading import check

with check("Exercise 1"):
    given1 = np.array([[3, 1, 4], [2, 7, 5]])
    got1 = np.asarray(ex1_t)
    assert got1.shape == (3, 2), (
        "E has 2 rows and 3 columns, so its transpose has 3 rows and 2 columns -- "
        f"yours has shape {got1.shape}"
    )
    assert np.allclose(got1, given1.T), (
        "each column of E should read across one row of your answer -- start by "
        "re-checking E's first column against your first row"
    )

### Exercise 2 — A column becomes a row

Build the column vector

$$
v = \begin{pmatrix} 5 \\ 2 \\ 9 \end{pmatrix}
$$

as a genuine two-dimensional array with 3 rows and 1 column, then flip it into a
row.

Define `ex2_col` — the column, shape `(3, 1)` — and `ex2_row` — its transpose,
shape `(1, 3)`. Section 1 warned about the shape that looks like a column and is
not; this exercise is where that warning gets tested.

<details><summary>Hint 1 — nudge</summary>

A flat array of three numbers has one axis and cannot be folded. A column vector
is a grid that happens to be one entry wide, so it needs a row per entry — three
rows, each holding a single number.

</details>
<details><summary>Hint 2 — approach</summary>

Nest each entry in its own list, `[[...], [...], [...]]`, or build the flat array
and reshape it to 3 rows by 1 column. Print `.shape` on the result and confirm it
has two numbers in it before you flip anything.

</details>

In [ ]:
# Exercise 2: Build v as a 3-by-1 column, then flip it into a row.
# Define: ex2_col, ex2_row

# TODO: your code here

In [ ]:
# Check Exercise 2 -- run after your attempt.
from lib.grading import check

with check("Exercise 2"):
    col2 = np.asarray(ex2_col)
    assert col2.shape == (3, 1), (
        "a genuine column is two-dimensional: 3 rows, 1 column. A flat "
        f"np.array([5, 2, 9]) has shape (3,) and cannot be flipped -- yours is {col2.shape}"
    )
    assert np.allclose(col2, [[5], [2], [9]]), (
        "build the column with the entries the prompt gives, in that order"
    )
    row2 = np.asarray(ex2_row)
    assert row2.shape == (1, 3), (
        "flipping a 3-by-1 gives a 1-by-3 -- still two-dimensional, one row of "
        f"three entries, but yours has shape {row2.shape}"
    )
    assert np.allclose(row2, col2.T), (
        "the row should list the column's entries top to bottom, left to right"
    )

### Exercise 3 — The product rule, by hand

Check property 3 on a pair of your own. With

$$
R = \begin{pmatrix} 1 & 2 \\ 3 & 0 \end{pmatrix},
\qquad
S = \begin{pmatrix} 4 & 1 \\ 2 & 5 \end{pmatrix}
$$

compute both sides of $(RS)^{T} = S^{T}R^{T}$ on paper, then type the two results
in separately so you can see them agree.

Define `ex3_prod_t` — the product RS, flipped — and `ex3_swapped` — the flipped
factors multiplied in the reversed order. Both are 2 by 2, and if your arithmetic
is sound they are the same matrix.

<details><summary>Hint 1 — nudge</summary>

The two sides are computed in genuinely different orders: one multiplies first and
flips afterwards, the other flips first and multiplies afterwards with the factors
swapped. Do them independently — copying one into the other proves nothing.

</details>
<details><summary>Hint 2 — approach</summary>

For the left side: multiply R by S the usual way (row of R against column of S),
then flip the 2 by 2 result. For the right side: write down S-transpose and
R-transpose, then multiply them in that order, S-transpose first. Compare the two
answers entry by entry before typing either one in.

</details>

In [ ]:
# Exercise 3: Verify (RS)^T = S^T R^T by hand on the given pair.
# Define: ex3_prod_t, ex3_swapped

# TODO: your code here

In [ ]:
# Check Exercise 3 -- run after your attempt.
from lib.grading import check

with check("Exercise 3"):
    r3 = np.array([[1, 2], [3, 0]])
    s3 = np.array([[4, 1], [2, 5]])
    left3 = np.asarray(ex3_prod_t)
    right3 = np.asarray(ex3_swapped)
    assert left3.shape == (2, 2) and right3.shape == (2, 2), (
        "both sides are 2 by 2 -- a 2-by-2 product stays 2 by 2 when flipped"
    )
    assert np.allclose(left3, (r3 @ s3).T), (
        "multiply R by S first, then flip the result -- re-check the top-left "
        "entry of the product: row 1 of R against column 1 of S"
    )
    assert np.allclose(right3, s3.T @ r3.T), (
        "the reversed product puts S-transpose first -- check you did not "
        "multiply R-transpose by S-transpose instead"
    )
    assert np.allclose(left3, right3), (
        "the whole point of the rule is that these two agree, so one of them has "
        "a slip in it -- recompute both and compare entry by entry"
    )

### Exercise 4 — The sum rule, by hand

Now property 4, on rectangles so that the shape change stays visible. With

$$
U = \begin{pmatrix} 2 & 5 & 1 \\ 0 & 3 & 4 \end{pmatrix},
\qquad
W = \begin{pmatrix} 1 & 0 & 6 \\ 7 & 2 & 3 \end{pmatrix}
$$

compute both sides of $(U + W)^{T} = U^{T} + W^{T}$ by hand.

Define `ex4_sum_t` — the sum U + W, flipped — and `ex4_t_sum` — the two flipped
matrices added together. Both have 3 rows and 2 columns.

<details><summary>Hint 1 — nudge</summary>

Adding matrices is entrywise, and both U and W have the same shape, so the sum has
that shape too. Whatever shape the sum has, its flip has the row and column counts
the other way round — which tells you the shape of both answers before you compute
either.

</details>
<details><summary>Hint 2 — approach</summary>

Left side: add the two matrices entry by entry to get a 2 by 3, then flip it into
a 3 by 2. Right side: flip U into a 3 by 2, flip W into a 3 by 2, then add those
two entry by entry. Two different routes, and property 4 claims one destination.

</details>

In [ ]:
# Exercise 4: Verify (U + W)^T = U^T + W^T by hand on the given pair.
# Define: ex4_sum_t, ex4_t_sum

# TODO: your code here

In [ ]:
# Check Exercise 4 -- run after your attempt.
from lib.grading import check

with check("Exercise 4"):
    u4 = np.array([[2, 5, 1], [0, 3, 4]])
    w4 = np.array([[1, 0, 6], [7, 2, 3]])
    left4 = np.asarray(ex4_sum_t)
    right4 = np.asarray(ex4_t_sum)
    assert left4.shape == (3, 2) and right4.shape == (3, 2), (
        "U and W are 2 by 3, so the sum is 2 by 3 and its flip is 3 by 2 -- "
        f"yours are {left4.shape} and {right4.shape}"
    )
    assert np.allclose(left4, (u4 + w4).T), (
        "add first, then flip -- re-check the entries of U + W before flipping"
    )
    assert np.allclose(right4, u4.T + w4.T), (
        "flip each matrix on its own, then add the two 3-by-2 results"
    )
    assert np.allclose(left4, right4), (
        "property 4 says these two routes land in the same place, so one of them "
        "has an arithmetic slip"
    )

### Exercise 5 — Symmetric or not

Classify each of these four matrices. Symmetric means the matrix equals its own
flip: every entry matches its mirror partner across the main diagonal.

$$
\text{(a)}\ \begin{pmatrix} 1 & 2 \\ 2 & 1 \end{pmatrix}
\qquad
\text{(b)}\ \begin{pmatrix} 0 & 3 \\ -3 & 0 \end{pmatrix}
$$

$$
\text{(c)}\ \begin{pmatrix} 4 & 0 & 1 \\ 0 & 5 & 2 \\ 1 & 2 & 6 \end{pmatrix}
\qquad
\text{(d)}\ \begin{pmatrix} 1 & 2 & 3 \\ 2 & 4 & 5 \\ 3 & 6 & 7 \end{pmatrix}
$$

Define `ex5_flags` — a list of four `True`/`False` values, one per matrix, in the
order (a), (b), (c), (d). Decide by eye; the check will tell you which ones to
look at again, not which way they go.

<details><summary>Hint 1 — nudge</summary>

The diagonal entries are their own mirror partners, so they can be anything at all
and never decide the question. Only the off-diagonal pairs matter — and a single
mismatched pair is enough to settle it.

</details>
<details><summary>Hint 2 — approach</summary>

For each matrix, walk the pairs: entry in row 1 column 2 against row 2 column 1,
then row 1 column 3 against row 3 column 1, then row 2 column 3 against row 3
column 2. A two by two has one pair to check; a three by three has three. Watch
the signs in (b) and the bottom-left corner of (d).

</details>

In [ ]:
# Exercise 5: Classify the four matrices as symmetric or not.
# Define: ex5_flags

# TODO: your code here

In [ ]:
# Check Exercise 5 -- run after your attempt.
from lib.grading import check

with check("Exercise 5"):
    mats5 = [
        np.array([[1, 2], [2, 1]]),
        np.array([[0, 3], [-3, 0]]),
        np.array([[4, 0, 1], [0, 5, 2], [1, 2, 6]]),
        np.array([[1, 2, 3], [2, 4, 5], [3, 6, 7]]),
    ]
    flags5 = list(ex5_flags)
    assert len(flags5) == 4, (
        f"one True or False per matrix, in the order (a), (b), (c), (d) -- "
        f"you gave {len(flags5)}"
    )
    want5 = [bool(np.allclose(m, m.T)) for m in mats5]
    off5 = [chr(ord("a") + i) for i in range(4) if bool(flags5[i]) != want5[i]]
    assert not off5, (
        "compare each entry against its mirror across the main diagonal -- "
        f"look again at {', '.join(off5)}"
    )

### Exercise 6 — Build a symmetric matrix

Section 3 gave two recipes that turn any matrix into a symmetric one. Apply both
to

$$
K = \begin{pmatrix} 3 & 1 \\ 8 & 5 \end{pmatrix}
$$

Define `ex6_sum` — the matrix K plus its own transpose — and `ex6_prod` — the
matrix K times its own transpose, with K on the left. Both are 2 by 2, and both
should come out equal to their own flips.

<details><summary>Hint 1 — nudge</summary>

The first recipe adds, which is entrywise and quick. The second multiplies, so the
usual row-against-column work applies — and the order matters, since the recipe is
K on the left and its flip on the right.

</details>
<details><summary>Hint 2 — approach</summary>

Write K-transpose down first; both recipes need it. Then add it to K entrywise for
the first answer, and multiply K by it for the second. Before moving on, check
each result against its own mirror: if a construction that is supposed to be
symmetric is not, the arithmetic went wrong somewhere.

</details>

In [ ]:
# Exercise 6: Apply both symmetric-building recipes to K.
# Define: ex6_sum, ex6_prod

# TODO: your code here

In [ ]:
# Check Exercise 6 -- run after your attempt.
from lib.grading import check

with check("Exercise 6"):
    k6 = np.array([[3, 1], [8, 5]])
    sum6 = np.asarray(ex6_sum)
    prod6 = np.asarray(ex6_prod)
    assert sum6.shape == (2, 2) and prod6.shape == (2, 2), (
        "K is 2 by 2, so both constructions are 2 by 2"
    )
    assert np.allclose(sum6, sum6.T) and np.allclose(prod6, prod6.T), (
        "both recipes are supposed to produce symmetric matrices -- if one of "
        "yours is not its own flip, recompute it"
    )
    assert np.allclose(sum6, k6 + k6.T), (
        "add K to its transpose entrywise -- the diagonal entries double"
    )
    assert np.allclose(prod6, k6 @ k6.T), (
        "multiply with K on the left and its transpose on the right -- check the "
        "top-left entry: row 1 of K against row 1 of K again"
    )

### Exercise 7 — Delete a row and a column

Take the 4 by 4 matrix

$$
N = \begin{pmatrix}
5 & 2 & 9 & 1 \\
3 & 8 & 4 & 6 \\
7 & 0 & 2 & 5 \\
1 & 6 & 3 & 9
\end{pmatrix}
$$

and form the minor left after deleting **row 2 and column 4**, counting from one
the way the textbooks do.

Define `ex7_minor` — the 3 by 3 matrix that survives. You may build it with
`np.delete` or type the surviving entries in yourself; either way, convert the
one-based instruction into whatever your tool expects before you touch an index.

<details><summary>Hint 1 — nudge</summary>

Deleting one row and one column from a 4 by 4 leaves a 3 by 3. The instruction
names what disappears, not what stays — the survivors are the other three rows and
the other three columns, in their original order.

</details>
<details><summary>Hint 2 — approach</summary>

`np.delete(N, i, axis=0)` drops a row and `np.delete(N, j, axis=1)` drops a
column, and both count from zero — so translate "row 2" and "column 4" before
passing them in. Apply one, then apply the other to the result. Check the shape of
what comes back before you trust it.

</details>

In [ ]:
# Exercise 7: Delete row 2 and column 4 (counting from one) from N.
# Define: ex7_minor

# TODO: your code here

In [ ]:
# Check Exercise 7 -- run after your attempt.
from lib.grading import check

with check("Exercise 7"):
    n7 = np.array([[5, 2, 9, 1],
                   [3, 8, 4, 6],
                   [7, 0, 2, 5],
                   [1, 6, 3, 9]])
    got7 = np.asarray(ex7_minor)
    assert got7.shape == (3, 3), (
        "deleting one row and one column from a 4 by 4 leaves a 3 by 3 -- "
        f"yours has shape {got7.shape}"
    )
    assert np.allclose(got7, np.delete(np.delete(n7, 1, axis=0), 3, axis=1)), (
        "textbook row 2 is position 1 and textbook column 4 is position 3 when "
        "counting from zero -- an off-by-one here deletes a neighbour instead"
    )

### Exercise 8 — Slice a block

Same idea, the other convention. Take

$$
V = \begin{pmatrix}
4 & 1 & 0 & 2 \\
6 & 3 & 5 & 7 \\
8 & 2 & 9 & 1 \\
0 & 5 & 3 & 6
\end{pmatrix}
$$

and pull out the 2 by 2 block formed by its **second and third rows** and its
**second and third columns**, counting from one.

Define `ex8_block` — that 2 by 2 block — using NumPy slicing, `V[r0:r1, c0:c1]`.
The translation is the exercise: rows described from one, inclusive, have to
become positions from zero with the stop excluded.

<details><summary>Hint 1 — nudge</summary>

NumPy counts rows from zero, so the row a textbook calls the second is the one
NumPy calls 1. And a slice stops **before** its second number, so covering two
positions means a range whose endpoints differ by two.

</details>
<details><summary>Hint 2 — approach</summary>

Work out the two positions you want in NumPy's counting, then write a slice that
starts at the first of them and stops one past the second. The same range works
for the columns here, since the prompt asks for the same two positions in both
directions. Confirm the result is 2 by 2 before trusting the entries.

</details>

In [ ]:
# Exercise 8: Slice out the block in rows 2-3 and columns 2-3 (counting from one).
# Define: ex8_block

# TODO: your code here

In [ ]:
# Check Exercise 8 -- run after your attempt.
from lib.grading import check

with check("Exercise 8"):
    v8 = np.array([[4, 1, 0, 2],
                   [6, 3, 5, 7],
                   [8, 2, 9, 1],
                   [0, 5, 3, 6]])
    got8 = np.asarray(ex8_block)
    assert got8.shape == (2, 2), (
        "two rows and two columns make a 2 by 2 -- a slice covering the wrong "
        f"number of positions shows up here first, and yours is {got8.shape}"
    )
    assert np.allclose(got8, v8[1:3, 1:3]), (
        "the second row counting from one is position 1 counting from zero, and "
        "the slice has to stop one past the last position you want"
    )

### Exercise 9 — Break a law that looks true

Here is a statement that holds for ordinary numbers and fails for matrices:

> If $AB = AC$ and A is not the zero matrix, then B = C.

Disprove it. Produce three 2 by 2 matrices with integer entries where A has at
least one nonzero entry, the two products $AB$ and $AC$ come out identical, and
yet B and C are different matrices.

Define `ex9_a`, `ex9_b`, `ex9_c` — your A, B and C. There are many valid answers;
any triple with those properties disproves the statement, which is the whole point
of a counterexample.

<details><summary>Hint 1 — nudge</summary>

For the products to agree, whatever makes B and C different has to be invisible
after A gets hold of them. So look for an A that throws information away — one
that flattens some part of every matrix it multiplies — and then let B and C
differ only in the part that gets thrown away.

</details>
<details><summary>Hint 2 — approach</summary>

A matrix with a row of zeros erases the corresponding row of its partner. Pick
such an A, then write B and C so that they agree everywhere except in the row A
erases. Multiply both products out by hand and confirm they match before you type
anything in.

</details>

In [ ]:
# Exercise 9: Build a counterexample to the cancellation law.
# Define: ex9_a, ex9_b, ex9_c

# TODO: your code here

In [ ]:
# Check Exercise 9 -- run after your attempt.
from lib.grading import check

with check("Exercise 9"):
    a9 = np.asarray(ex9_a, dtype=float)
    b9 = np.asarray(ex9_b, dtype=float)
    c9 = np.asarray(ex9_c, dtype=float)
    assert a9.shape == (2, 2) and b9.shape == (2, 2) and c9.shape == (2, 2), (
        "all three matrices should be 2 by 2"
    )
    assert not np.allclose(a9, np.zeros((2, 2))), (
        "the zero matrix is the one case the statement already excludes, so it "
        "cannot serve as the counterexample"
    )
    assert not np.allclose(b9, c9), (
        "B and C have to be genuinely different, or there is nothing to disprove"
    )
    assert np.allclose(a9 @ b9, a9 @ c9), (
        "the two products have to come out identical -- try an A that erases "
        "whatever makes B and C differ"
    )

### Exercise 10 — A property suite in NumPy

Close the notebook by testing the properties instead of trusting them. Use

$$
A = \begin{pmatrix} 1 & 2 & 0 \\ 0 & 3 & 1 \\ 4 & 1 & 2 \end{pmatrix},
\qquad
B = \begin{pmatrix} 2 & 1 & 3 \\ 1 & 0 & 2 \\ 5 & 4 & 1 \end{pmatrix}
$$

Define `ex10_a` and `ex10_b` as those two matrices, and `ex10_checks` — a
dictionary with exactly these six keys, each holding the `True` or `False` that
`np.allclose` returns for the claim beside it, using c = 3 for the scalar one:

- `"double"` — flipping A twice gives A back
- `"scalar"` — flipping 3A gives 3 times the flip of A
- `"product"` — flipping AB gives the flip of B times the flip of A
- `"sum"` — flipping A + B gives the flip of A plus the flip of B
- `"symmetric"` — the matrix A plus the flip of A equals its own flip
- `"swapped"` — flipping AB gives the flip of A times the flip of B

Five of the six come out one way and one does not. Compute every entry rather than
assuming, and compare with `np.allclose` rather than `==`.

<details><summary>Hint 1 — nudge</summary>

Five of these keys restate properties this notebook proved, so their values follow
from the theory. The sixth is the version of the product rule with the order left
alone — the exact mistake section 2 warned about — and the whole reason it is in
the list is that it does not behave like the others.

</details>
<details><summary>Hint 2 — approach</summary>

Build the dictionary key by key, and let each value be a single `np.allclose(...)`
call comparing the two sides of that claim. Nothing needs a loop and nothing needs
a helper function; a literal dictionary with six entries is the whole answer.

</details>

In [ ]:
# Exercise 10: Test all five transpose properties, plus the order mistake, on A and B.
# Define: ex10_a, ex10_b, ex10_checks

# TODO: your code here

In [ ]:
# Check Exercise 10 -- run after your attempt.
from lib.grading import check

with check("Exercise 10"):
    a10 = np.array([[1, 2, 0], [0, 3, 1], [4, 1, 2]])
    b10 = np.array([[2, 1, 3], [1, 0, 2], [5, 4, 1]])
    want10 = {
        "double": np.allclose(a10.T.T, a10),
        "scalar": np.allclose((3 * a10).T, 3 * a10.T),
        "product": np.allclose((a10 @ b10).T, b10.T @ a10.T),
        "sum": np.allclose((a10 + b10).T, a10.T + b10.T),
        "symmetric": np.allclose(a10 + a10.T, (a10 + a10.T).T),
        "swapped": np.allclose((a10 @ b10).T, a10.T @ b10.T),
    }
    got_a10 = np.asarray(ex10_a)
    got_b10 = np.asarray(ex10_b)
    assert got_a10.shape == (3, 3) and got_b10.shape == (3, 3), (
        "the prompt gives two 3-by-3 matrices, so both should have shape (3, 3) -- "
        f"yours are {got_a10.shape} and {got_b10.shape}"
    )
    assert np.allclose(got_a10, a10), "build A exactly as the prompt gives it"
    assert np.allclose(got_b10, b10), "build B exactly as the prompt gives it"
    assert set(ex10_checks) == set(want10), (
        "the dictionary needs exactly the six keys the prompt lists -- missing "
        f"{sorted(set(want10) - set(ex10_checks))}, unexpected "
        f"{sorted(set(ex10_checks) - set(want10))}"
    )
    off10 = sorted(k for k in want10 if bool(ex10_checks[k]) != bool(want10[k]))
    assert not off10, (
        "run both sides of each claim through np.allclose rather than reasoning "
        f"about it -- these came out the other way: {', '.join(off10)}"
    )

### Solutions

Worked answers to all ten, in order. Run them if you want to compare, but run your
own attempt and its check first — reading a solution feels like learning and is
not.

In [ ]:
# --- Exercise 1 ---
ex1_t = np.array([[3, 2],
                  [1, 7],
                  [4, 5]])                  # E's columns, read across
print('ex1_t =')
print(ex1_t)

# --- Exercise 2 ---
ex2_col = np.array([[5],
                    [2],
                    [9]])                   # 2-D: 3 rows, 1 column
ex2_row = ex2_col.T
print('\ncolumn shape:', ex2_col.shape, ' row shape:', ex2_row.shape)
print(ex2_row)

# --- Exercise 3 ---
ex3_prod_t = np.array([[8, 12],
                       [11, 3]])            # R @ S, then flipped
ex3_swapped = np.array([[8, 12],
                        [11, 3]])           # S-transpose @ R-transpose
print('\nex3 sides agree?', np.allclose(ex3_prod_t, ex3_swapped))

# --- Exercise 4 ---
ex4_sum_t = np.array([[3, 7],
                      [5, 5],
                      [7, 7]])              # (U + W) flipped
ex4_t_sum = np.array([[3, 7],
                      [5, 5],
                      [7, 7]])              # U-transpose + W-transpose
print('ex4 sides agree?', np.allclose(ex4_sum_t, ex4_t_sum))

# --- Exercise 5 ---
ex5_flags = [True, False, True, False]      # (b) mirrors with the sign flipped; (d) pairs 5 with 6
print('ex5_flags =', ex5_flags)

# --- Exercise 6 ---
ex6_sum = np.array([[3, 1], [8, 5]]) + np.array([[3, 1], [8, 5]]).T
ex6_prod = np.array([[3, 1], [8, 5]]) @ np.array([[3, 1], [8, 5]]).T
print('\nex6_sum =')
print(ex6_sum)
print('ex6_prod =')
print(ex6_prod)

# --- Exercise 7 ---
ex7_minor = np.delete(
    np.delete(np.array([[5, 2, 9, 1],
                        [3, 8, 4, 6],
                        [7, 0, 2, 5],
                        [1, 6, 3, 9]]), 1, axis=0),   # textbook row 2 -> position 1
    3, axis=1,                                        # textbook column 4 -> position 3
)
print('\nex7_minor =')
print(ex7_minor)

# --- Exercise 8 ---
ex8_block = np.array([[4, 1, 0, 2],
                      [6, 3, 5, 7],
                      [8, 2, 9, 1],
                      [0, 5, 3, 6]])[1:3, 1:3]        # rows 2-3, columns 2-3 from one
print('ex8_block =')
print(ex8_block)

# --- Exercise 9 ---
ex9_a = np.array([[2, 0], [0, 0]])          # doubles row 1, erases row 2
ex9_b = np.array([[1, 3], [5, 2]])
ex9_c = np.array([[1, 3], [0, 7]])          # differs from B only in the erased row
print('\nproducts equal?', np.allclose(ex9_a @ ex9_b, ex9_a @ ex9_c),
      ' B equals C?', np.allclose(ex9_b, ex9_c))

# --- Exercise 10 ---
ex10_a = np.array([[1, 2, 0], [0, 3, 1], [4, 1, 2]])
ex10_b = np.array([[2, 1, 3], [1, 0, 2], [5, 4, 1]])
ex10_checks = {
    "double": np.allclose(ex10_a.T.T, ex10_a),
    "scalar": np.allclose((3 * ex10_a).T, 3 * ex10_a.T),
    "product": np.allclose((ex10_a @ ex10_b).T, ex10_b.T @ ex10_a.T),
    "sum": np.allclose((ex10_a + ex10_b).T, ex10_a.T + ex10_b.T),
    "symmetric": np.allclose(ex10_a + ex10_a.T, (ex10_a + ex10_a.T).T),
    "swapped": np.allclose((ex10_a @ ex10_b).T, ex10_a.T @ ex10_b.T),
}
for name, verdict in ex10_checks.items():
    print(f'{name:>10}: {verdict}')

## Where this shows up in quantum computing

Four connections, and the first one is the reason this notebook exists.

**The dagger is this flip plus one more step.** In
[`00-prereqs/notebooks/02-linear-algebra-for-quantum.ipynb`](../../00-prereqs/notebooks/02-linear-algebra-for-quantum.ipynb)
you meet `M.conj().T`, the **conjugate transpose**, written $M^{\dagger}$ and read
"M dagger". It is exactly the flip you just learned, followed by flipping the sign
of every imaginary part. Nothing in this notebook has to be relearned there: the
order still reverses, $(AB)^{\dagger} = B^{\dagger}A^{\dagger}$, sums and scalars
still pass through, and for a matrix whose entries are ordinary real numbers there
is nothing to conjugate, so the dagger simply **is** the transpose.

**Order reversal is why you undo a circuit backwards.** Apply gate $U_1$ and then
gate $U_2$ and the combined operation is the product $U_2 U_1$. Undoing it means
taking the dagger of that product, which by the reversal rule is
$U_1^{\dagger} U_2^{\dagger}$ — the gates undone in the opposite order from the
order they were applied. That is a literal recipe for inverting a circuit: reverse
the list of gates and dagger each one. Algorithms in section 03 lean on it
constantly, under the name "uncompute".

**Symmetric is the real cousin of Hermitian.** A matrix with $M = M^{\dagger}$ is
called **Hermitian**, and everything physically measurable — energy, spin along an
axis, the molecular Hamiltonians in section 05 — is represented by one, because
that condition is what forces the measured values to come out as real numbers. Set
the imaginary parts to zero and the condition collapses to $M = M^{T}$: symmetric.
Every intuition you built in section 3 above carries over unchanged.

**You have already sliced a gate out of a bigger gate.** The 4 by 4 matrix in
section 4 was not decorative — it is **CNOT**, the two-qubit gate that section 01
builds entanglement with:

$$
\mathrm{CNOT} = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{pmatrix}
$$

Its top-left 2 by 2 block is the identity: when the control qubit is 0, the target
is left alone. Its bottom-right 2 by 2 block is `[[0, 1], [1, 0]]`, the X gate —
the quantum NOT — so when the control is 1, the target flips. Every controlled
operation has that block form, identity in one corner and the gate being
controlled in the other, and reading it is a submatrix extraction like the one you
did in exercise 8.

One last note on the two laws that failed. Quantum gates are unitary, which means
every one of them can be undone exactly — and where inverses exist, cancellation
comes back: for gates, $UA = UB$ really does force A = B, and a product of gates
is never the zero matrix. The failures of section 5 live outside the gate set, in
measurement and noise, which is precisely where quantum information genuinely does
get destroyed.

## Summary

- The **transpose** flips a matrix across its main diagonal: rows become columns,
  the diagonal stays put, and $(M^{T})_{ij} = M_{ji}$. In NumPy it is `M.T`.
- **Shapes trade places.** A 2 by 3 flips into a 3 by 2, and a genuine column
  vector needs two dimensions — flipping a flat `(3,)` array silently does
  nothing at all.
- **Five properties.** Flipping twice returns the original; scalars pass through;
  sums pass through; a product **reverses**, $(AB)^{T} = B^{T}A^{T}$; and
  $A = A^{T}$ is the definition of **symmetric**.
- The reversal is forced by the shapes, not chosen by convention, and it is the
  single most common transpose mistake. $A^{T}B^{T}$ is usually a different
  matrix, and often not even a legal product.
- **Two recipes always produce symmetry**: $A + A^{T}$ for a square A, and
  $AA^{T}$ for any A at all — both proved in one line from the properties above.
- **Submatrices come in two conventions.** Textbooks delete a named row and column
  counting from one; NumPy keeps a sliced block counting from zero with the stop
  excluded. Say which one you are in before you write an index.
- **Two laws fail.** $AB = AC$ does not give B = C, and $AB = 0$ does not give a
  zero factor. Both failures trace to the same fact: you cannot divide by a
  matrix, and some matrices destroy information.

**You finished notebook 4, the last of this module's four.** See
[`../GUIDE.md`](../GUIDE.md) for the whole arc, then go on to
[`00-prereqs`](../../00-prereqs/GUIDE.md), where every operation you just learned
picks up a quantum name and the transpose grows a conjugation.